# 07 - Analysis & Visualization

Loads all `eval/results/{checkpoint}/{scores,timing}.json` files plus `outputs/training_metrics.json`, builds the master results table, computes SFT/quantization deltas and efficiency metrics, and generates the 6 figures from the project plan.

Runs locally (no GPU needed) -- just reads the JSON results produced by notebooks 02-06.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

REPO_DIR = Path("..")
RESULTS_DIR = REPO_DIR / "eval" / "results"
OUTPUTS_DIR = REPO_DIR / "outputs"
FIGURES_DIR = REPO_DIR / "figures"
REPORTS_DIR = REPO_DIR / "reports"
FIGURES_DIR.mkdir(exist_ok=True)

import sys
sys.path.insert(0, str(REPO_DIR / "src"))
from efficient_slm.evaluation.compare import build_results_dataframe, compute_delta_table
from efficient_slm.evaluation.metrics import calculate_efficiency, calculate_quality_per_param

## Load results

In [ ]:
CHECKPOINTS = ["base", "sft_r4", "sft_r8", "sft_r16", "quantized_r4", "quantized_r8", "quantized_r16"]
BENCHMARKS = ["mmlu", "arc", "gsm8k", "hellaswag"]

scores_by_checkpoint, timing_by_checkpoint = {}, {}
for checkpoint in CHECKPOINTS:
    scores_path = RESULTS_DIR / checkpoint / "scores.json"
    timing_path = RESULTS_DIR / checkpoint / "timing.json"
    if scores_path.exists():
        with open(scores_path) as f:
            scores_by_checkpoint[checkpoint] = json.load(f)["scores"]
    if timing_path.exists():
        with open(timing_path) as f:
            timing_by_checkpoint[checkpoint] = json.load(f)

training_metrics_path = OUTPUTS_DIR / "training_metrics.json"
training_metrics = {}
if training_metrics_path.exists():
    with open(training_metrics_path) as f:
        training_metrics = json.load(f)

print("Checkpoints with scores:", list(scores_by_checkpoint.keys()))
print("Checkpoints with timing:", list(timing_by_checkpoint.keys()))

missing = [c for c in CHECKPOINTS if c not in scores_by_checkpoint]
if missing:
    print(f"\nMissing scores for: {missing} -- run 02_baseline_eval.ipynb / 06_evaluation.ipynb for these first.")

## Master results table

In [ ]:
results_df = build_results_dataframe(scores_by_checkpoint, timing_by_checkpoint, training_metrics)
results_df

## Deltas: SFT impact and quantization cost

In [ ]:
benchmark_cols = [b for b in BENCHMARKS if b in results_df.columns]
delta_df = compute_delta_table(results_df, benchmark_cols)
delta_df

## Efficiency metrics

In [ ]:
results_df["avg_score"] = results_df[benchmark_cols].mean(axis=1)
results_df["quality_per_vram"] = results_df.apply(lambda r: calculate_efficiency(r["vram_gb"], r["avg_score"]), axis=1)
results_df["quality_per_param_million"] = results_df.apply(
    lambda r: calculate_quality_per_param(r["trainable_params"], r["avg_score"]), axis=1
)
results_df[["avg_score", "vram_gb", "quality_per_vram", "trainable_params", "quality_per_param_million"]]

## Figure 1: Quality Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_df = results_df[benchmark_cols].dropna(how="all")
plot_df.plot(kind="bar", ax=ax)
ax.set_ylabel("Score")
ax.set_title("Quality Comparison Across Checkpoints")
ax.legend(title="Benchmark")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_quality_comparison.png", dpi=150)
plt.show()

## Figure 2: Efficiency Profile (VRAM vs Score)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
plot_df = results_df.dropna(subset=["vram_gb", "avg_score"])
sizes = (plot_df["throughput_tokens_per_sec"].fillna(20) * 5).clip(lower=20)
ax.scatter(plot_df["vram_gb"], plot_df["avg_score"], s=sizes)
for name, row in plot_df.iterrows():
    label = f"{name}\n{row['quality_per_vram']:.1f} pts/GB"
    ax.annotate(label, (row["vram_gb"], row["avg_score"]), fontsize=8, textcoords="offset points", xytext=(6, 6))
ax.set_xlabel("VRAM (GB)")
ax.set_ylabel("Average score")
ax.set_title("Efficiency Profile: VRAM vs Quality (bubble size = throughput)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_efficiency_profile.png", dpi=150)
plt.show()

## Figure 3: VRAM-Latency Trade-off

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
plot_df = results_df.dropna(subset=["vram_gb", "latency_ms_per_token"])
ax.scatter(plot_df["vram_gb"], plot_df["latency_ms_per_token"], s=80)
for name, row in plot_df.iterrows():
    ax.annotate(name, (row["vram_gb"], row["latency_ms_per_token"]), fontsize=8, textcoords="offset points", xytext=(6, 6))
ax.set_xlabel("VRAM (GB)")
ax.set_ylabel("Latency (ms/token)")
ax.set_title("VRAM-Latency Trade-off")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_vram_latency.png", dpi=150)
plt.show()

## Figure 4: Quality Loss Due to Quantization

In [ ]:
if not delta_df.empty:
    fig, ax = plt.subplots(figsize=(9, 6))
    pivot = delta_df.pivot(index="benchmark", columns="rank", values="delta_quant")
    pivot.plot(kind="bar", ax=ax)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_ylabel("Score delta (quantized - SFT)")
    ax.set_title("Quality Loss Due to Quantization")
    ax.legend(title="LoRA rank")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "04_quantization_loss.png", dpi=150)
    plt.show()
else:
    print("No delta data available yet -- need both sft_r* and quantized_r* results.")

## Figure 5: Scaling with LoRA Rank

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ranks = [4, 8, 16]
for series_name, prefix in [("SFT", "sft_r"), ("Quantized", "quantized_r")]:
    ys = [results_df.loc[f"{prefix}{r}", "avg_score"] if f"{prefix}{r}" in results_df.index else None for r in ranks]
    if any(y is not None for y in ys):
        ax.plot(ranks, ys, marker="o", label=series_name)
if "base" in results_df.index:
    ax.axhline(results_df.loc["base", "avg_score"], color="gray", linestyle="--", label="Base")
ax.set_xlabel("LoRA rank")
ax.set_ylabel("Average score")
ax.set_title("Scaling with LoRA Rank")
ax.set_xticks(ranks)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_lora_rank_scaling.png", dpi=150)
plt.show()

## Figure 6: Model Size vs Quality

In [ ]:
size_comparison_path = OUTPUTS_DIR / "size_comparison.json"
size_comparison = {}
if size_comparison_path.exists():
    with open(size_comparison_path) as f:
        size_comparison = json.load(f)

rows = []
for name, row in results_df.iterrows():
    size_gb = None
    if name.startswith("sft_r"):
        size_gb = (size_comparison.get(name.replace("sft_", "")) or {}).get("merged_gb")
    elif name.startswith("quantized_r"):
        size_gb = (size_comparison.get(name.replace("quantized_", "")) or {}).get("quantized_gb")
    if size_gb is not None:
        rows.append({"checkpoint": name, "size_gb": size_gb, "avg_score": row["avg_score"], "vram_gb": row["vram_gb"]})

if rows:
    size_df = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(size_df["size_gb"], size_df["avg_score"], s=80)
    for _, row in size_df.iterrows():
        label = f"{row['checkpoint']}\n({row['vram_gb']:.1f}GB VRAM)"
        ax.annotate(label, (row["size_gb"], row["avg_score"]), fontsize=8, textcoords="offset points", xytext=(6, 6))
    ax.set_xlabel("Model size on disk (GB)")
    ax.set_ylabel("Average score")
    ax.set_title("Model Size vs Quality")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "06_model_size_quality.png", dpi=150)
    plt.show()
else:
    print("No size_comparison.json found yet -- run 04_merge_quantize.ipynb first.")

## Save tables for the report

In [ ]:
REPORTS_DIR.mkdir(exist_ok=True)
results_df.to_csv(REPORTS_DIR / "results_table.csv")
delta_df.to_csv(REPORTS_DIR / "delta_table.csv", index=False)
print("Saved reports/results_table.csv and reports/delta_table.csv")